# 12 — Merge the final Commission manifest

This notebook performs **only the merge step**.

Rows sharing the same `candidate_to_merge` value are consolidated into one Commission-decision row. No row is deleted here, even when `candidate_to_delete` is populated. The delete/relevance step is handled separately in notebook 13.

For each merge group, distinct nonblank values are retained in their original order and joined with ` | ` when they differ.

Input:
`input/final_commission_manifest_manual_changes.xlsx`

Output:
`output/final_manifest/final_commission_manifest_merged.xlsx`

In [ ]:
from pathlib import Path
from datetime import date, datetime
import re

import pandas as pd
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter

## 1. Paths and input

In [ ]:
def project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for directory in (start, *start.parents):
        if (directory / "input").is_dir():
            return directory
    raise FileNotFoundError(
        "No input/ directory found in this directory or its parents. "
        "Set ROOT explicitly to the ec_cjeu project path."
    )


ROOT = project_root()

INPUT = ROOT / "input" / "final_commission_manifest_manual_changes.xlsx"
OUTPUT = ROOT / "output" / "final_manifest" / "final_commission_manifest_merged.xlsx"

if not INPUT.is_file():
    raise FileNotFoundError(
        f"Place the manually edited final Commission manifest here: {INPUT}"
    )

print("Input:", INPUT)
print("Output:", OUTPUT)

## 2. Read the manually edited manifest

The `files` sheet is used when present; otherwise the first sheet is read.

`candidate_to_merge` determines which rows belong to the same underlying Commission decision. A blank merge ID is treated as a standalone row.

`candidate_to_delete` is preserved but **not acted upon** in this notebook.

In [ ]:
book = pd.ExcelFile(INPUT)
sheet = "files" if "files" in book.sheet_names else book.sheet_names[0]

files = pd.read_excel(
    INPUT,
    sheet_name=sheet,
    dtype=object,
    keep_default_na=False,
)
files.columns = [str(column).strip() for column in files.columns]

if files.columns.duplicated().any():
    raise ValueError("Duplicate column names found in the input sheet.")

required = {"candidate_to_merge", "candidate_to_delete"}
missing = required - set(files.columns)
if missing:
    raise ValueError(
        f"The input sheet {sheet!r} is missing columns: {sorted(missing)}"
    )

print(
    f"Read {len(files):,} rows from {sheet!r}; "
    f"{len(files.columns)} columns."
)

## 3. Merge rows

In [ ]:
def merge_id(value, excel_row):
    if value is None or str(value).strip() == "":
        return f"standalone_row_{excel_row}"

    if isinstance(value, (int, float)) and not isinstance(value, bool):
        if pd.isna(value):
            return f"standalone_row_{excel_row}"
        if float(value).is_integer():
            return str(int(value))

    label = str(value).strip()

    if re.fullmatch(r"\d+\.0+", label):
        return label.split(".")[0]

    return label


working = files.copy()
working["input_excel_row"] = working.index + 2
working["candidate_to_merge"] = [
    merge_id(value, int(row))
    for value, row in zip(
        working["candidate_to_merge"],
        working["input_excel_row"],
    )
]

SEPARATOR = " | "


def display_value(value):
    if value is None:
        return ""

    if not isinstance(value, (list, dict, set)) and pd.isna(value):
        return ""

    if isinstance(value, (datetime, date, pd.Timestamp)):
        return value.strftime("%Y-%m-%d")

    if isinstance(value, float) and value.is_integer():
        return str(int(value))

    return str(value).strip()


def combine_values(values):
    distinct = []
    seen = set()

    for value in values:
        rendered = display_value(value)

        if rendered and rendered not in seen:
            distinct.append(rendered)
            seen.add(rendered)

    result = SEPARATOR.join(distinct)

    if len(result) > 32767:
        raise ValueError(
            "A merged Excel cell exceeds the 32,767-character limit."
        )

    return result


columns = list(files.columns)

grouped = working.groupby(
    "candidate_to_merge",
    sort=False,
    dropna=False,
)

records = []

for group_id, group in grouped:
    record = {
        "candidate_to_merge": group_id,
    }

    for column in columns:
        if column == "candidate_to_merge":
            continue

        try:
            record[column] = combine_values(group[column])
        except ValueError as exc:
            raise ValueError(
                f"Merge ID {group_id!r}, column {column!r}: {exc}"
            ) from exc

    record["merged_file_count"] = len(group)
    records.append(record)


merged = pd.DataFrame.from_records(
    records,
    columns=columns + ["merged_file_count"],
)

# Sort by the earliest available date for the merge group.
date_column = next(
    (
        column
        for column in ("case_date", "decision_date", "date")
        if column in working.columns
    ),
    None,
)

if date_column and len(merged):
    dates = working.assign(
        _parsed_date=pd.to_datetime(
            working[date_column],
            errors="coerce",
            dayfirst=False,
        )
    )

    earliest = dates.groupby(
        "candidate_to_merge",
        sort=False,
    )["_parsed_date"].min()

    merged["_sort_date"] = merged["candidate_to_merge"].map(earliest)

    merged = (
        merged.sort_values(
            ["_sort_date", "candidate_to_merge"],
            kind="stable",
            na_position="last",
        )
        .drop(columns="_sort_date")
    )

merged = merged.reset_index(drop=True)

print(
    f"Created {len(merged):,} merged decisions "
    f"from {len(working):,} file rows."
)
print(
    "Groups with multiple files:",
    int((merged["merged_file_count"] > 1).sum()) if len(merged) else 0,
)

## 4. Save the merged manifest

The output contains a single `merged_manifest` sheet.

No relevance filtering or deletion is performed here.

In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
pending = OUTPUT.with_name(OUTPUT.stem + ".pending.xlsx")

with pd.ExcelWriter(pending, engine="openpyxl") as writer:
    merged.to_excel(
        writer,
        sheet_name="merged_manifest",
        index=False,
    )

    ws = writer.sheets["merged_manifest"]
    ws.freeze_panes = "C2"
    ws.auto_filter.ref = ws.dimensions

    for cell in ws[1]:
        cell.fill = PatternFill("solid", fgColor="19334D")
        cell.font = Font(color="FFFFFF", bold=True)

    for index, column in enumerate(merged.columns, start=1):
        ws.column_dimensions[get_column_letter(index)].width = min(
            65,
            max(17, len(str(column)) + 3),
        )

pending.replace(OUTPUT)

print("Wrote:", OUTPUT)